# Probabilistic Diagnostic Reasoning Demo (v4)

This version implements your pointers:

### A) Warnings policy
- If a warning is present in the **initial input**, it is treated as evidence and will **not** be suggested as a check.
- If a warning is **not** present in the input, the system *may* suggest a `PRC_CHECK_WARN_*` procedure if it is useful.

### B) Upstream root causes
- Upstream causes are generated dynamically each session (keyword/tag overlap).
- You can “check for upstream causes” by:
  1) Looking at `dynamic_upstream_causes` printed at session start (these are not in the error-code list),
  2) Optionally using `cause_relations` (`upstream_of`) when present in the KB.

### C) Hard elimination on negative evidence
- Some checks are *targeted* to specific root causes.
- If a targeted check returns a **negative** result (does NOT support the targeted cause), we **eliminate** that cause:
  - set its posterior probability to **0**
  - keep it eliminated for the rest of the session.

This is implemented via KB fields:
- `targets_root_cause_ids`
- `supporting_outcomes`

---

## What do epsilon, lambda_time, lambda_parts mean?
- **epsilon**: if all causes have probability < epsilon → escalate
- **lambda_time**, **lambda_parts**: trade-off weights in the procedure score:

  Score = IG - lambda_time·time_min - lambda_parts·parts_eur


In [1]:
import json, math, uuid, datetime
from pathlib import Path

KB_PATH = Path(r"C:/Project/CPP_CORE/kb/kb_ink_system_pump_tube_v4.json")
kb = json.loads(KB_PATH.read_text(encoding="utf-8"))

root_causes = {rc["root_cause_id"]: rc for rc in kb["catalogs"]["root_causes"]}
obs_catalog = {o["observation_id"]: o for o in kb["catalogs"]["observations"]}
procedures = {p["procedure_id"]: p for p in kb["catalogs"]["procedures"]}
error_models = {m["error_code"]: m for m in kb["catalogs"]["error_code_models"]}

likelihoods = kb["parameters"].get("likelihoods", {})
action_success = kb["parameters"].get("action_success", {})

cfg = kb["inference_config"]
EPSILON = cfg["stop_condition"]["epsilon_default"]
REPAIR_RECOMMEND = cfg["repair_recommendation"]["recommend_threshold"]
REPAIR_CONSIDER = cfg["repair_recommendation"]["consider_threshold"]

UP_CFG = cfg.get("upstream_generation", {})
EXP_CFG = cfg.get("expensive_repair_policy", {})
EXP_PARTS_TH = EXP_CFG.get("parts_eur_threshold", 500)
CONFIRM_MAP = EXP_CFG.get("confirming_procedures_by_repair", {})
CONFIRM_FALLBACK = EXP_CFG.get("fallback_if_no_mapping", "any_check_or_test")

WARN_CFG = cfg.get("warning_policy", {})
WARNING_OBS_IDS = set(WARN_CFG.get("warning_observation_ids", ["OBS_WARNING_290005_SEEN","OBS_WARNING_290020_SEEN"]))
WARNING_PROC_PREFIX = WARN_CFG.get("warning_check_procedure_prefix", "PRC_CHECK_WARN_")

CAUSE_RELATIONS = kb["catalogs"].get("cause_relations", [])

print("Loaded KB:", kb["meta"]["name"])
print("Error codes:", len(error_models), "| Causes:", len(root_causes), "| Procedures:", len(procedures))


Loaded KB: Ink system pump/tube KB (v4, warning policy + elimination rules + cause relations)
Error codes: 9 | Causes: 31 | Procedures: 44


In [2]:
def normalize(dist: dict) -> dict:
    s = sum(dist.values())
    if s <= 0:
        n = len(dist)
        return {k: 1.0/n for k in dist} if n else {}
    return {k: v/s for k, v in dist.items()}

def entropy(dist: dict) -> float:
    H = 0.0
    for p in dist.values():
        if p > 1e-12:
            H -= p * math.log(p, 2)
    return H

def get_likelihood(obs_id: str, cause_id: str, value: str) -> float:
    vals = obs_catalog[obs_id]["values"]
    table = likelihoods.get(obs_id, {}).get(cause_id)
    if table is None:
        return 1.0 / len(vals)
    return float(table.get(value, 1.0 / len(vals)))

def update_belief(belief: dict, obs_id: str, value: str) -> dict:
    updated = {}
    for c, p in belief.items():
        updated[c] = p * get_likelihood(obs_id, c, value)
    return normalize(updated)

def action_outcome_likelihood(action_id: str, cause_id: str, resolved_value: str) -> float:
    table = action_success.get(action_id, {}).get(cause_id)
    if table is None:
        return 0.45 if resolved_value == "yes" else 0.55
    return float(table.get(resolved_value, 0.5))

def update_after_repair(belief: dict, action_id: str, resolved_value: str) -> dict:
    updated = {}
    for c, p in belief.items():
        updated[c] = p * action_outcome_likelihood(action_id, c, resolved_value)
    return normalize(updated)

def top_k(belief: dict, k: int = 10):
    return sorted(belief.items(), key=lambda kv: kv[1], reverse=True)[:k]

def now_utc_iso():
    return datetime.datetime.now(datetime.UTC).isoformat().replace("+00:00", "Z")


In [3]:
def context_keywords(error_code: str, symptoms_yes: list[str], warnings: dict) -> set[str]:
    model = error_models[error_code]
    txt = (model.get("description","") + " " + " ".join(symptoms_yes)).lower()
    for oid, val in warnings.items():
        if val == "yes":
            txt += " " + oid.lower()
    keys = set()
    for k in UP_CFG.get("keywords", []):
        if k in txt:
            keys.add(k)
    return keys

def is_upstream(cause_id: str, error_code: str) -> bool:
    return cause_id not in set(error_models[error_code]["candidate_root_causes"])

def upstream_of(cause_id: str) -> list[str]:
    # returns downstream causes if KB contains upstream_of relations
    outs = []
    for r in CAUSE_RELATIONS:
        if r.get("relation") == "upstream_of" and r.get("from_cause_id") == cause_id:
            outs.append(r.get("to_cause_id"))
    return [c for c in outs if c]

def propose_upstream_causes(error_code: str, symptoms_yes: list[str], warnings: dict, max_n: int = 15) -> list[str]:
    listed = set(error_models[error_code]["candidate_root_causes"])
    keys = context_keywords(error_code, symptoms_yes, warnings)

    scored = []
    for cid, rc in root_causes.items():
        if cid in listed:
            continue
        label = (rc.get("label","") or "").lower()
        tags = set(rc.get("tags") or [])
        score = 0
        for k in keys:
            if k in label:
                score += 2
            if k.replace(" ", "_") in tags:
                score += 2
        if warnings.get("OBS_WARNING_290005_SEEN") == "yes" and "pump_tube" in tags:
            score += 3
        if warnings.get("OBS_WARNING_290020_SEEN") == "yes" and "ink_expired" in tags:
            score += 3
        if score > 0:
            scored.append((score, cid))

    scored.sort(reverse=True)
    return [cid for _, cid in scored[:max_n]]

def build_initial_prior(error_code: str, symptoms_yes: list[str], warnings: dict, alpha: float = 0.90):
    ec_prior = normalize({cid: float(p) for cid, p in error_models[error_code]["priors_root_cause_given_error"].items()})
    belief = {cid: 0.0 for cid in root_causes.keys()}

    upstream = propose_upstream_causes(error_code, symptoms_yes, warnings, max_n=UP_CFG.get("max_upstream_causes",15))
    other_mass = max(0.0, 1.0 - alpha)
    if upstream:
        base = other_mass / len(upstream)
        for cid in upstream:
            belief[cid] += base
    else:
        other = [cid for cid in root_causes.keys() if cid not in ec_prior]
        if other:
            base = other_mass / len(other)
            for cid in other:
                belief[cid] += base

    for cid, p in ec_prior.items():
        belief[cid] += alpha * p

    return normalize(belief), upstream


In [4]:
def recompute_posterior(prior: dict, evidence_state: dict, eliminated: set) -> dict:
    belief = prior.copy()
    for obs_id, value in evidence_state.items():
        belief = update_belief(belief, obs_id, value)
    # hard elimination
    for cid in eliminated:
        belief[cid] = 0.0
    return normalize(belief)

def apply_elimination_from_procedure(proc_id: str, outcome: str, eliminated: set) -> None:
    proc = procedures[proc_id]
    targets = proc.get("targets_root_cause_ids") or []
    supporting = proc.get("supporting_outcomes") or []
    if not targets or not supporting:
        return
    # If outcome does NOT support the targeted causes -> eliminate targets
    if outcome not in supporting:
        for cid in targets:
            eliminated.add(cid)


In [5]:
def predicted_outcome_distribution(belief: dict, proc_id: str) -> dict:
    proc = procedures[proc_id]
    vals = proc["outcomes"]
    dist = {v: 0.0 for v in vals}

    if proc["kind"] == "repair":
        for v in vals:
            for c, p in belief.items():
                dist[v] += p * action_outcome_likelihood(proc_id, c, v)
    else:
        obs_id = proc["produces_observation"]
        for v in vals:
            for c, p in belief.items():
                dist[v] += p * get_likelihood(obs_id, c, v)

    return normalize(dist)

def expected_entropy_after(belief: dict, proc_id: str) -> float:
    proc = procedures[proc_id]
    out_dist = predicted_outcome_distribution(belief, proc_id)
    expH = 0.0
    for v, pv in out_dist.items():
        if pv <= 1e-12:
            continue
        if proc["kind"] == "repair":
            post = update_after_repair(belief, proc_id, v)
        else:
            post = update_belief(belief, proc["produces_observation"], v)
        expH += pv * entropy(post)
    return expH

def score_procedure(belief: dict, proc_id: str, lambda_time: float, lambda_parts: float) -> dict:
    H0 = entropy(belief)
    expH = expected_entropy_after(belief, proc_id)
    ig = H0 - expH
    cost = procedures[proc_id]["cost"]
    score = ig - lambda_time * cost["time_min"] - lambda_parts * cost["parts_eur"]
    return {"procedure_id": proc_id, "score": score, "ig": ig, "time_min": cost["time_min"], "parts_eur": cost["parts_eur"]}

def confirm_done_for(repair_id: str, tried: set) -> bool:
    required = CONFIRM_MAP.get(repair_id)
    if required:
        return any(pid in tried for pid in required)
    if CONFIRM_FALLBACK == "any_check_or_test":
        return any(procedures[pid]["kind"] in ("check","test") for pid in tried)
    return False

def warning_check_allowed(proc_id: str, evidence_state: dict) -> bool:
    # Only suggest warning check if its observation isn't already provided
    proc = procedures[proc_id]
    obs_id = proc.get("produces_observation")
    if obs_id in WARNING_OBS_IDS:
        return obs_id not in evidence_state
    return True

def prerequisite_ok(proc_id: str, evidence_state: dict) -> bool:
    # Keep the earlier practical rule: don't do syringe tube-flow unless valves are known open
    if proc_id == "PRC_CHECK_INK_TUBE_FLOW":
        return evidence_state.get("OBS_INK_VALVES_OPEN") == "yes"
    return True

def suggest_next(belief: dict, error_code: str, tried: set, evidence_state: dict, lambda_time: float, lambda_parts: float) -> list:
    cand = error_models[error_code]["candidate_procedure_ids"]
    scored = []
    for pid in cand:
        if pid in tried:
            continue
        if pid.startswith(WARNING_PROC_PREFIX) and not warning_check_allowed(pid, evidence_state):
            continue
        if not prerequisite_ok(pid, evidence_state):
            continue
        proc = procedures[pid]
        if proc["kind"] == "repair" and proc.get("is_expensive", False):
            if not confirm_done_for(pid, tried):
                continue
        scored.append(score_procedure(belief, pid, lambda_time=lambda_time, lambda_parts=lambda_parts))
    return sorted(scored, key=lambda d: d["score"], reverse=True)


In [6]:
from pathlib import Path

def exhausted(belief: dict, epsilon: float = EPSILON) -> bool:
    return all(p < epsilon for p in belief.values())

def run_session(
    error_code: str,
    symptoms_yes=None,
    initial_warnings=None,
    initial_counters=None,
    max_steps: int = 15,
    log_dir: str = "logs",
    lambda_time: float = 0.02,
    lambda_parts: float = 0.005
):
    if error_code not in error_models:
        raise ValueError(f"Unknown error code: {error_code}")

    symptoms_yes = symptoms_yes or []
    initial_warnings = initial_warnings or {}
    initial_counters = initial_counters or {}

    session_id = str(uuid.uuid4())
    ts = now_utc_iso()
    tried = set()
    history = []
    evidence_state = {}
    eliminated = set()

    # Initial prior + upstream
    prior, upstream = build_initial_prior(
        error_code, symptoms_yes, initial_warnings,
        alpha=UP_CFG.get("alpha_error_code_mass", 0.90)
    )

    # Add symptoms as evidence-state (binary yes)
    for s in symptoms_yes:
        oid = "OBS_SYM_" + "".join([ch if ch.isalnum() else "_" for ch in s.lower()]).upper()[:50]
        oid = oid.replace("__","_")
        if oid not in obs_catalog:
            obs_catalog[oid] = {"observation_id": oid, "label": f"Symptom: {s}", "type": "binary", "values": ["yes", "no"]}
        evidence_state[oid] = "yes"
        history.append({"type": "symptom", "observation_id": oid, "value": "yes", "note": s})

    # Warnings as initial evidence if provided
    for oid, val in initial_warnings.items():
        evidence_state[oid] = val
        history.append({"type": "warning", "observation_id": oid, "value": val})

    # Counters
    for oid, val in initial_counters.items():
        evidence_state[oid] = val
        history.append({"type": "counter", "observation_id": oid, "value": val})

    belief = recompute_posterior(prior, evidence_state, eliminated)

    print("\nSESSION", session_id, "| error_code", error_code)
    print("epsilon:", EPSILON, "| recommend:", REPAIR_RECOMMEND, "| consider:", REPAIR_CONSIDER)
    print("lambda_time:", lambda_time, "| lambda_parts:", lambda_parts)
    if upstream:
        print("Dynamic upstream causes added:", len(upstream))
        # Show how to check upstream causes:
        print("Sample upstream cause labels:")
        for cid in upstream[:5]:
            print("  -", cid, "|", root_causes[cid]["label"])

    for step in range(1, max_steps + 1):
        print("\n--- Step", step, "---")
        top = top_k(belief, 8)
        for cid, p in top:
            tag = " (UPSTREAM)" if is_upstream(cid, error_code) else ""
            print(f"{p:0.3f}  {cid}{tag} | {root_causes[cid]['label']}")

        if exhausted(belief):
            print("\nAll hypotheses < epsilon. ESCALATE.")
            history.append({"type": "stop", "reason": "exhausted"})
            break

        suggestions = suggest_next(belief, error_code, tried, evidence_state, lambda_time, lambda_parts)
        if not suggestions:
            print("\nNo more allowed procedures. ESCALATE.")
            history.append({"type": "stop", "reason": "no_procedures"})
            break

        best = suggestions[0]
        pid = best["procedure_id"]
        proc = procedures[pid]

        top_c, top_p = top[0]
        snap = {
            "top_cause_id": top_c,
            "top_cause_p": top_p,
            "top_cause_label": root_causes[top_c]["label"],
            "selected_procedure_id": pid,
            "selected_procedure_label": proc["label"],
            "selected_procedure_kind": proc["kind"]
        }

        print(f"\nNEXT: {pid} | {proc['label']}")
        print(f"  kind={proc['kind']} score={best['score']:.4f} IG={best['ig']:.4f} time={best['time_min']}m parts=€{best['parts_eur']}")
        print("  outcomes:", proc["outcomes"])
        if pid.startswith(WARNING_PROC_PREFIX):
            print("  NOTE: warning check suggested because warning not provided initially.")
        if proc.get("targets_root_cause_ids"):
            print("  targeted causes:")
            for tc in proc["targets_root_cause_ids"]:
                print("   -", tc, "|", root_causes.get(tc, {}).get("label","?"))

        outcome = input("Enter outcome value: ").strip()
        if outcome not in proc["outcomes"]:
            print("Invalid outcome. Try again.")
            continue

        tried.add(pid)

        # Apply hard elimination rule if this is a targeted procedure
        apply_elimination_from_procedure(pid, outcome, eliminated)

        if proc["kind"] == "repair":
            belief = update_after_repair(belief, pid, outcome)
            # Apply eliminations after repair update too
            for cid in eliminated:
                belief[cid] = 0.0
            belief = normalize(belief)

            history.append({"type": "repair", "procedure_id": pid, "outcome": outcome, "recommended_snapshot": snap, "eliminated_now": list(eliminated)})

            if outcome == "yes":
                print("\nResolved=yes. STOP.")
                history.append({"type": "stop", "reason": "resolved"})
                break
        else:
            obs_id = proc["produces_observation"]
            evidence_state[obs_id] = outcome  # overwrite semantics
            belief = recompute_posterior(prior, evidence_state, eliminated)
            history.append({"type": proc["kind"], "procedure_id": pid, "observation_id": obs_id, "value": outcome,
                            "recommended_snapshot": snap, "eliminated_now": list(eliminated)})

    ground_truth = None
    if history and history[-1].get("reason") == "resolved":
        for h in reversed(history):
            if h.get("type") == "repair" and h.get("outcome") == "yes":
                snap = h.get("recommended_snapshot")
                ground_truth = {
                    "mode": "recommended_snapshot_at_resolving_repair",
                    "root_cause_id": snap["top_cause_id"],
                    "label": snap["top_cause_label"],
                    "p_at_action_time": snap["top_cause_p"],
                    "repair_procedure_id": snap["selected_procedure_id"],
                    "repair_label": snap["selected_procedure_label"]
                }
                break

    log_entry = {
        "session_id": session_id,
        "timestamp_utc": ts,
        "error_code": error_code,
        "symptoms_yes": symptoms_yes,
        "initial_warnings": initial_warnings,
        "initial_counters": initial_counters,
        "dynamic_upstream_causes": upstream,
        "eliminated_causes_final": sorted(list(eliminated)),
        "evidence_state_final": evidence_state,
        "history": history,
        "final_top_causes": [{"cause_id": cid, "p": p, "label": root_causes[cid]["label"]} for cid, p in top_k(belief, 10)],
        "epsilon": EPSILON,
        "lambda_time": lambda_time,
        "lambda_parts": lambda_parts,
        "ground_truth": ground_truth
    }

    Path(log_dir).mkdir(parents=True, exist_ok=True)
    log_path = Path(log_dir) / "diagnostic_sessions.jsonl"
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

    print("\nLog saved to:", log_path.resolve())
    print("Ground truth:", ground_truth)
    return log_entry


## Example run

Try two runs:
1) Provide warnings upfront → warning checks will NOT be suggested.
2) Omit a warning from input → the model may suggest a warning check if useful.


In [ ]:
error_code = "250012"
symptoms_yes = [""]

# Try toggling these:
# initial_warnings = {
#     "OBS_WARNING_290005_SEEN": "yes",
#     "OBS_WARNING_290020_SEEN": "no"
# }
initial_warnings = {}  # uncomment to see warning checks become eligible

initial_counters = {}

log_entry = run_session(
    error_code,
    symptoms_yes=symptoms_yes,
    initial_warnings=initial_warnings,
    initial_counters=initial_counters,
    max_steps=12,
    lambda_time=0.02,
    lambda_parts=0.005
)



SESSION 6d6753ee-9457-48d9-b4ab-421e05ecebfe | error_code 250012
epsilon: 0.02 | recommend: 0.8 | consider: 0.75
lambda_time: 0.02 | lambda_parts: 0.005
Dynamic upstream causes added: 15
Sample upstream cause labels:
  - RC_CLOSED_INK_VALVES_AT_PRINTHEAD | Closed ink valves at printhead
  - RC_BLOCKAGE_IN_INK_TUBE_FROM_3_WAY_VALVE_TO | Blockage in ink tube from 3-way valve to Printhead
  - RC_FAULTY_PRINTHEAD | Faulty printhead
  - RC_BLOCKED_PRINTHEAD_VALVE | Blocked printhead valve
  - RC_INK_TUBE_OUT_OF_PARAMETER | Ink tube out of parameter

--- Step 1 ---
0.900  RC_PRESSURE_FAILURE_INSIDE_THE_PRINTHEAD_DU | Pressure failure inside the printhead due to a blocked air-tube
0.007  RC_BLOCKED_PRINTHEAD_VALVE (UPSTREAM) | Blocked printhead valve
0.007  RC_INK_LEAKAGE (UPSTREAM) | Ink Leakage
0.007  RC_FAULTY_PRINTHEAD (UPSTREAM) | Faulty printhead
0.007  RC_FAULTY_ELECTRICAL_CONNECTION_OF_INK_LEAK (UPSTREAM) | Faulty electrical connection of Ink Leak sensor
0.007  RC_FAULT_INK_LEAK_SENS

: 